# Phased Allocation with Handover

Same 10 employees as `simple_allocation.ipynb`, but the project now runs in **two phases**. Each phase favours a different specialist on skill alone, so solving the phases *independently* (`handover=0`) churns the team between them. Turning on the **handover** reward keeps a stable core across consecutive phases.

In [1]:
from optimizer.models import (
    PersonInput, SkillLevel, Seniority,
    ProjectInput, ProjectPhase, SkillRequirement,
    AssignmentWeights,
)
from optimizer.adapters.pulp_solver import PuLPTeamAssignmentSolver

## Employees

Skills are scored 0–5. Each role has a strong primary skill (4.5) and weak secondary ones. A few people (e.g. `ml_3`, `ds_5`) are dual-skilled — those are the ones handover can profitably retain across phases.

In [2]:
ML  = 'machine_learning'
DS  = 'data_science'
DEV = 'software_development'

people = [
    # ── ML engineers ──────────────────────────────────────────────────────
    PersonInput(id='ml_1', seniority=Seniority.SENIOR, years_of_experience=6, skills=[SkillLevel(id=ML, level=4.5), SkillLevel(id=DS, level=2.0)]),
    PersonInput(id='ml_2', seniority=Seniority.MID,    years_of_experience=4, skills=[SkillLevel(id=ML, level=4.0), SkillLevel(id=DS, level=1.5)]),
    PersonInput(id='ml_3', seniority=Seniority.LEAD,   years_of_experience=8, skills=[SkillLevel(id=ML, level=5.0), SkillLevel(id=DS, level=3.0)]),

    # ── Data scientists ────────────────────────────────────────────────
    PersonInput(id='ds_1', seniority=Seniority.SENIOR, years_of_experience=5, skills=[SkillLevel(id=DS, level=4.5), SkillLevel(id=ML, level=2.5)]),
    PersonInput(id='ds_2', seniority=Seniority.MID,    years_of_experience=3, skills=[SkillLevel(id=DS, level=4.0), SkillLevel(id=ML, level=1.5)]),
    PersonInput(id='ds_3', seniority=Seniority.SENIOR, years_of_experience=7, skills=[SkillLevel(id=DS, level=5.0), SkillLevel(id=ML, level=2.0)]),
    PersonInput(id='ds_4', seniority=Seniority.JUNIOR, years_of_experience=2, skills=[SkillLevel(id=DS, level=3.5), SkillLevel(id=ML, level=1.0)]),
    PersonInput(id='ds_5', seniority=Seniority.LEAD,   years_of_experience=9, skills=[SkillLevel(id=DS, level=4.5), SkillLevel(id=ML, level=3.0)]),

    # ── Developers ───────────────────────────────────────────────────
    PersonInput(id='dev_1', seniority=Seniority.SENIOR, years_of_experience=5, skills=[SkillLevel(id=DEV, level=4.5)]),
    PersonInput(id='dev_2', seniority=Seniority.MID,    years_of_experience=3, skills=[SkillLevel(id=DEV, level=4.0)]),
]

## Project

One project, `gamma`, run in two consecutive phases of 3 slots each:

| Phase    | Slots | Skill emphasis |
|----------|-------|----------------|
| modeling | 3     | machine_learning |
| analysis | 3     | data_science |

On skill alone the modeling phase pulls ML engineers and the analysis phase pulls data scientists — two largely different teams.

In [3]:
project_gamma = ProjectInput(
    id='gamma',
    phases=[
        ProjectPhase(id='modeling', n_slots=3, skill_requirements=[SkillRequirement(id=ML, min_level=4.0)]),
        ProjectPhase(id='analysis', n_slots=3, skill_requirements=[SkillRequirement(id=DS, min_level=4.0)]),
    ],
)

## Solve

Identical weights both times — the only difference is `handover`. With `handover=0` each phase is optimised on its own; with `handover>0` the phases are solved jointly and retaining a person across the two phases earns a bonus.

In [4]:
solver = PuLPTeamAssignmentSolver()

weights_independent = AssignmentWeights(performance=0.5, chemistry=0.1, growth=0.2, cost=0.2, handover=0.0)
weights_handover    = AssignmentWeights(performance=0.5, chemistry=0.1, growth=0.2, cost=0.2, handover=0.4)

result_independent = solver.solve(project_gamma, people, weights_independent)
result_handover    = solver.solve(project_gamma, people, weights_handover)

## Results

`retained` is the set of people kept across both phases — the metric handover is meant to maximise.

In [5]:
def show(result):
    print(f'Project {result.project_id}  (score={result.score})')
    by_phase = {}
    for m in result.members:
        by_phase.setdefault(m.phase_id, []).append(m.person_id)
    for phase_id, ids in by_phase.items():
        print(f'  {phase_id:<10} {sorted(ids)}')
    retained = set.intersection(*(set(ids) for ids in by_phase.values()))
    print(f'  retained   {sorted(retained)}  ({len(retained)} kept across phases)')
    print()

print('=== handover = 0 (phases solved independently) ===')
show(result_independent)

print('=== handover = 0.4 (joint solve, team kept stable) ===')
show(result_handover)

=== handover = 0 (phases solved independently) ===
Project gamma  (score=3.87)
  modeling   ['ml_1', 'ml_2', 'ml_3']
  analysis   ['ds_1', 'ds_2', 'ds_3']
  retained   []  (0 kept across phases)

=== handover = 0.4 (joint solve, team kept stable) ===
Project gamma  (score=4.5225)
  modeling   ['ds_1', 'ds_5', 'ml_3']
  analysis   ['ds_1', 'ds_5', 'ml_3']
  retained   ['ds_1', 'ds_5', 'ml_3']  (3 kept across phases)

